In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import dill
import json
import sys
import os
from scipy import spatial
import matplotlib.pyplot as plt

In [6]:
result_path = 'results/NO_concentration_data_source/'
dict_path = 'reshaped_ev_points_dict.csv'

In [7]:
concentration_data =[]

for i in range(500):
    file_path = result_path + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 2].isin([3303])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data.append(concentrations)
concentration_data_np = np.stack(concentration_data, axis = 1)

In [8]:
fig = go.Figure(go.Scatter(y=concentration_data_np[0], name = 'overall',
                          mode = 'lines',
                          line = dict(width= 5, color= 'red')))

fig.update_xaxes(title = 'Time [ms]')
fig.update_yaxes(title = 'NO Concentration [pM]',range = [-5*10**(-12), 140*10**(-12)])
fig.update_layout(title_text = 'Concentration at source')

fig.show()

In [3]:
ev_points = pd.read_csv(os.path.join(result_path,dict_path))
cluster = np.unique(ev_points['cluster'].values)

In [4]:
#rand_clusters = np.random.choice(cluster, size=, replace=False)
rand_clusters = [71]
clusters_df = ev_points[ev_points['cluster'].isin(rand_clusters)]
ev_points_n_clusters = clusters_df[['ev_points_id', 'x', 'y', 'z']].values

In [5]:
concentration_data_clusters =[]
epsilon = 1e-10
conc_max = 4_000_000
for i in range(500):
    file_path = result_path + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    log_concentrations = np.log10(concentrations + epsilon)
    concentration_data_clusters.append(log_concentrations)

In [ ]:
frames = []
for i in range(500):
    frame = go.Frame(
        data=[
            go.Scatter3d(
                x=ev_points_n_clusters[:, 1],
                y=ev_points_n_clusters[:, 2],
                z=ev_points_n_clusters[:, 3],
                text = ev_points_n_clusters[:,0],
                mode='markers',
                marker=dict(
                    size=3,
                    color=concentration_data_clusters[i],  # Set color to the log-transformed concentrations
                    colorscale='Viridis',  # Color scale
                    cmin=0,  # Set minimum value for the color scale
                    cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                    colorbar=dict(title='Log10 Concentration'),
                    opacity=0.8
                )
            )
        ],
        name=str(i)
    )
    frames.append(frame)

# Create the initial figure
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=ev_points_n_clusters[:, 1],
            y=ev_points_n_clusters[:, 2],
            z=ev_points_n_clusters[:, 3],
            text = ev_points_n_clusters[:,0],
            mode='markers',
            marker=dict(
                size=3,
                color=np.zeros(len(ev_points_n_clusters)),  # Default color (zeros)
                colorscale='Viridis',
                cmin=0,  # Set minimum value for the color scale
                cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                colorbar=dict(title='Log10 Concentration'),
                opacity=0.8
            )
        )
    ],
    layout=go.Layout(
        updatemenus=[
            {
                'buttons': [
                    {
                        'args': [None, {'frame': {'duration': 100, 'redraw': True}, 'fromcurrent': True}],
                        'label': 'Play',
                        'method': 'animate'
                    },
                    {
                        'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                        'label': 'Pause',
                        'method': 'animate'
                    }
                ],
                'direction': 'left',
                'pad': {'r': 10, 't': 87},
                'showactive': True,
                'type': 'buttons',
                'x': 0.1,
                'xanchor': 'right',
                'y': 0,
                'yanchor': 'top'
            }
        ],
        sliders=[{
            'active': 0,
            'steps': [
                {
                    'label': str(i),
                    'method': 'animate',
                    'args': [
                        [str(i)],
                        {'mode': 'immediate', 'transition': {'duration': 300}}
                    ]
                }
                for i in range(500)
            ]
        }]
    )
)

# Add the frames to the figure
fig.frames = frames

fig.write_html('3dplot_NO_diffusion.html')

# Show the figure
fig.show()

In [155]:
concentration_data_clusters =[]
conc_max = 4_000_000
for i in range(500):
    file_path = result_path + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data_clusters.append(concentrations)

In [ ]:
np.nonzero(matched_conc.iloc[:,0].values==133420)

matched_conc.iloc[:,0].values==133420

In [ ]:
id = 34280
concentration_data_clusters_np = np.array(concentration_data_clusters)
concentration_plot = concentration_data_clusters_np[:,3801]

fig = go.Figure(go.Scatter(y=concentration_plot, name = 'overall',
                          mode = 'lines',
                          line = dict(width= 5, color= 'red')))

fig.update_xaxes(title = 'Time [ms]')
fig.update_yaxes(title = 'NO Concentration [pM]')
fig.update_layout(title_text = 'Concentration at receiver')

fig.show()

In [153]:
result_path_1 = 'results/NO_concentration_data_2/'
concentration_data_clusters_1 =[]
conc_max = 4_000_000
for i in range(500):
    file_path = result_path_1 + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data_clusters_1.append(concentrations)

In [ ]:
concentration_data_clusters_np_1 = np.array(concentration_data_clusters_1)
concentration_plot = concentration_data_clusters_np_1[:,3801]

fig = go.Figure(go.Scatter(y=concentration_plot, name = 'overall',
                          mode = 'lines',
                          line = dict(width= 5, color= 'red')))

fig.update_xaxes(title = 'Time [ms]')
fig.update_yaxes(title = 'NO Concentration [pM]')
fig.update_layout(title_text = 'Concentration at receiver')

fig.show()

In [ ]:
ev_points_id = ev_points['ev_points_id'].values
r_max = 15
nNOS_ev_points = []
for id_cluster in cluster:
            # loop on the receiver
            for evpoint_id in ev_points_id:
                df_row = ev_points.loc[evpoint_id+1]
                # loop on the sources
                if df_row['cluster']==id_cluster:
                    ev_point_coordinates = np.array([df_row['x'],df_row['y'],df_row['z']])
                    nNOS = []
                    for evpoint_id_1 in ev_points_id:
                        df_row_1 =  ev_points.loc[evpoint_id_1+1]
                        if df_row_1['cluster']==id_cluster:
                            nNOS_coordinates = np.array([df_row_1['x'],df_row_1['y'],df_row_1['z']])                   
                            # distance evaluation
                            d = spatial.distance.euclidean(nNOS_coordinates, ev_point_coordinates)
                            # check on relevant distance value
                            if d < r_max:
                                # lists update
                                nNOS.append(df_row_1['ev_points_id'])
                    nNOS_ev_points.append(nNOS)

In [ ]:
np.mean([len(nNOS_ev_points[i]) for i in range(len(nNOS_ev_points))])

In [16]:
result_path_noise = 'results/NO_concentration_data_noise/'
concentration_data_noise =[]
epsilon = 1e-10
conc_max = 4_000_000
for i in range(500):
    file_path = result_path_noise + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    log_concentrations_noise = np.log10(concentrations + epsilon)
    concentration_data_noise.append(log_concentrations_noise)

In [ ]:
frames = []
for i in range(500):
    frame = go.Frame(
        data=[
            go.Scatter3d(
                x=ev_points_n_clusters[:, 1],
                y=ev_points_n_clusters[:, 2],
                z=ev_points_n_clusters[:, 3],
                text = ev_points_n_clusters[:,0],
                mode='markers',
                marker=dict(
                    size=3,
                    color=concentration_data_noise[i],  # Set color to the log-transformed concentrations
                    colorscale='Viridis',  # Color scale
                    cmin=0,  # Set minimum value for the color scale
                    cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                    colorbar=dict(title='Log10 Concentration'),
                    opacity=0.8
                )
            )
        ],
        name=str(i)
    )
    frames.append(frame)

# Create the initial figure
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=ev_points_n_clusters[:, 1],
            y=ev_points_n_clusters[:, 2],
            z=ev_points_n_clusters[:, 3],
            text = ev_points_n_clusters[:,0],
            mode='markers',
            marker=dict(
                size=3,
                color=np.zeros(len(ev_points_n_clusters)),  # Default color (zeros)
                colorscale='Viridis',
                cmin=0,  # Set minimum value for the color scale
                cmax=np.log10(conc_max + epsilon),  # Set maximum value for the color scale
                colorbar=dict(title='Log10 Concentration'),
                opacity=0.8
            )
        )
    ],
    layout=go.Layout(
        updatemenus=[
            {
                'buttons': [
                    {
                        'args': [None, {'frame': {'duration': 100, 'redraw': True}, 'fromcurrent': True}],
                        'label': 'Play',
                        'method': 'animate'
                    },
                    {
                        'args': [[None], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate', 'transition': {'duration': 0}}],
                        'label': 'Pause',
                        'method': 'animate'
                    }
                ],
                'direction': 'left',
                'pad': {'r': 10, 't': 87},
                'showactive': True,
                'type': 'buttons',
                'x': 0.1,
                'xanchor': 'right',
                'y': 0,
                'yanchor': 'top'
            }
        ],
        sliders=[{
            'active': 0,
            'steps': [
                {
                    'label': str(i),
                    'method': 'animate',
                    'args': [
                        [str(i)],
                        {'mode': 'immediate', 'transition': {'duration': 300}}
                    ]
                }
                for i in range(500)
            ]
        }]
    )
)

# Add the frames to the figure
fig.frames = frames

fig.write_html('3dplot_NO_diffusion_noise.html')

# Show the figure
fig.show()

In [18]:

concentration_data_noise_1 =[]
conc_max = 4_000_000
for i in range(500):
    file_path = result_path_noise + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data_noise_1.append(concentrations)
concentration_data_noise_np = np.array(concentration_data_noise_1)

In [19]:
max_value_index = np.argmax(concentration_data_noise_np)

# Get the row and column index of the max value
row, col = np.unravel_index(max_value_index, concentration_data_noise_np.shape)

In [ ]:
col

In [ ]:
id = 1501

concentration_plot = concentration_data_noise_np[:,id]

fig = go.Figure(go.Scatter(y=concentration_plot, name = 'overall',
                          mode = 'lines',
                          line = dict(width= 5, color= 'red')))

fig.update_xaxes(title = 'Time [ms]')
fig.update_yaxes(title = 'NO Concentration [pM]')
fig.update_layout(title_text = 'Concentration at receiver')

fig.show()

In [ ]:
ev_points.iloc[:,0].values[1501]

In [27]:
result_path = 'results/NO_concentration_data_source_1/'

In [28]:
concentration_data =[]

for i in range(500):
    file_path = result_path + f'NO_concentration_t_{i}.csv'
    df_conc = pd.read_csv(file_path)
    matched_conc = df_conc[df_conc.iloc[:, 0].isin(ev_points_n_clusters[:, 0])]
    concentrations = matched_conc.iloc[:, 1].values
    concentration_data.append(concentrations)
concentration_data_np = np.stack(concentration_data, axis = 1)

In [30]:
max_value_index = np.argmax(concentration_data_np)

# Get the row and column index of the max value
row, col = np.unravel_index(max_value_index, concentration_data_np.shape)

In [ ]:
id = row

concentration_plot = concentration_data_np

fig = go.Figure(go.Scatter(y=concentration_plot, name = 'overall',
                          mode = 'lines',
                          line = dict(width= 5, color= 'red')))

fig.update_xaxes(title = 'Time [ms]')
fig.update_yaxes(title = 'NO Concentration [pM]')
fig.update_layout(title_text = 'Concentration at source')

fig.show()